In [1]:
# Import and paths
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv


In [6]:
from google.colab import files

uploaded = files.upload()

Saving PRSA_Data_Aotizhongxin_20130301-20170228.csv to PRSA_Data_Aotizhongxin_20130301-20170228.csv


In [8]:
import pandas as pd
import glob
import os

csv_files = glob.glob("/content/*.csv")

print("Uploaded CSV files:")
for file in csv_files:
    print(os.path.basename(file))

Uploaded CSV files:
PRSA_Data_Aotizhongxin_20130301-20170228.csv


In [9]:
file_path = next(
    file for file in csv_files
    if "Aotizhongxin" in os.path.basename(file)
)

raw_df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", raw_df.shape)

Dataset loaded successfully!
Shape: (35064, 18)


In [10]:
raw_df["datetime"] = pd.to_datetime(
    raw_df[["year", "month", "day", "hour"]]
)

raw_df = raw_df.set_index("datetime")

print(raw_df.head())

                     No  year  month  day  hour  PM2.5  PM10   SO2   NO2  \
datetime                                                                   
2013-03-01 00:00:00   1  2013      3    1     0    4.0   4.0   4.0   7.0   
2013-03-01 01:00:00   2  2013      3    1     1    8.0   8.0   4.0   7.0   
2013-03-01 02:00:00   3  2013      3    1     2    7.0   7.0   5.0  10.0   
2013-03-01 03:00:00   4  2013      3    1     3    6.0   6.0  11.0  11.0   
2013-03-01 04:00:00   5  2013      3    1     4    3.0   3.0  12.0  12.0   

                        CO    O3  TEMP    PRES  DEWP  RAIN   wd  WSPM  \
datetime                                                                
2013-03-01 00:00:00  300.0  77.0  -0.7  1023.0 -18.8   0.0  NNW   4.4   
2013-03-01 01:00:00  300.0  77.0  -1.1  1023.2 -18.2   0.0    N   4.7   
2013-03-01 02:00:00  300.0  73.0  -1.1  1023.5 -18.2   0.0  NNW   5.6   
2013-03-01 03:00:00  300.0  72.0  -1.4  1024.5 -19.4   0.0   NW   3.1   
2013-03-01 04:00:00  300.0  7

In [11]:
for lag in [1, 3, 6, 12, 24]:
    raw_df[f"PM2.5_lag_{lag}h"] = raw_df["PM2.5"].shift(lag)

In [12]:
for window in [3, 6, 12, 24]:
    raw_df[f"PM2.5_roll_mean_{window}h"] = raw_df["PM2.5"].rolling(window).mean()
    raw_df[f"PM2.5_roll_std_{window}h"] = raw_df["PM2.5"].rolling(window).std()

In [13]:
raw_df["hour"] = raw_df.index.hour
raw_df["dayofweek"] = raw_df.index.dayofweek
raw_df["month"] = raw_df.index.month

In [14]:
raw_df["hour_sin"] = np.sin(2 * np.pi * raw_df["hour"] / 24)
raw_df["hour_cos"] = np.cos(2 * np.pi * raw_df["hour"] / 24)

raw_df["month_sin"] = np.sin(2 * np.pi * raw_df["month"] / 12)
raw_df["month_cos"] = np.cos(2 * np.pi * raw_df["month"] / 12)

In [15]:
wd_dummies = pd.get_dummies(raw_df["wd"], prefix="wd")
raw_df = pd.concat([raw_df, wd_dummies], axis=1)

In [16]:
FEATURE_COLUMNS = [
    "PM10","SO2","NO2","CO","O3","TEMP","PRES","DEWP","RAIN","WSPM", "PM2.5_lag_1h","PM2.5_lag_3h","PM2.5_lag_6h","PM2.5_lag_12h","PM2.5_lag_24h", "PM2.5_roll_mean_3h","PM2.5_roll_std_3h", "PM2.5_roll_mean_6h","PM2.5_roll_std_6h",
    "PM2.5_roll_mean_12h","PM2.5_roll_std_12h", "PM2.5_roll_mean_24h","PM2.5_roll_std_24h", "hour","dayofweek","month", "hour_sin","hour_cos","month_sin","month_cos", "wd_ENE","wd_ESE","wd_N","wd_NE","wd_NNE","wd_NNW","wd_NW", "wd_S","wd_SE","wd_SSE","wd_SSW","wd_SW", "wd_W","wd_WNW","wd_WSW"
]

In [17]:
for col in FEATURE_COLUMNS:
    if col not in raw_df.columns:
        raw_df[col] = 0

In [18]:
processed_df = raw_df[FEATURE_COLUMNS].dropna()
processed_df.head(), processed_df.shape

(                     PM10   SO2   NO2     CO    O3  TEMP    PRES  DEWP  RAIN  \
 datetime                                                                       
 2013-03-02 00:00:00  24.0  24.0  44.0  500.0  44.0  -0.4  1031.0 -17.6   0.0   
 2013-03-02 01:00:00  17.0  21.0  36.0  400.0  50.0  -1.0  1031.3 -17.3   0.0   
 2013-03-02 02:00:00  13.0  20.0  37.0  400.0  47.0  -1.5  1030.9 -16.9   0.0   
 2013-03-02 03:00:00   9.0  13.0  34.0  400.0  52.0  -1.4  1030.6 -17.6   0.0   
 2013-03-02 04:00:00   7.0  18.0  43.0  400.0  43.0  -1.5  1030.8 -17.7   0.0   
 
                      WSPM  ...  wd_NNW  wd_NW   wd_S  wd_SE  wd_SSE  wd_SSW  \
 datetime                   ...                                                
 2013-03-02 00:00:00   1.4  ...   False  False  False  False   False   False   
 2013-03-02 01:00:00   1.1  ...   False  False  False  False   False   False   
 2013-03-02 02:00:00   1.7  ...   False  False  False  False   False   False   
 2013-03-02 03:00:00   1.4  ...

In [19]:
output_path = OUTPUT_DIR / "dongsi_ready_for_prediction.csv"

processed_df.to_csv(output_path)

print("Inference-ready dataset saved:")
print(output_path)


Inference-ready dataset saved:
../data/processed/dongsi_ready_for_prediction.csv


In [20]:
print("Dataset shape:", raw_df.shape)
print("Station:", raw_df["station"].unique())
print("Columns:", raw_df.columns.tolist())

Dataset shape: (35064, 52)
Station: ['Aotizhongxin']
Columns: ['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station', 'PM2.5_lag_1h', 'PM2.5_lag_3h', 'PM2.5_lag_6h', 'PM2.5_lag_12h', 'PM2.5_lag_24h', 'PM2.5_roll_mean_3h', 'PM2.5_roll_std_3h', 'PM2.5_roll_mean_6h', 'PM2.5_roll_std_6h', 'PM2.5_roll_mean_12h', 'PM2.5_roll_std_12h', 'PM2.5_roll_mean_24h', 'PM2.5_roll_std_24h', 'dayofweek', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'wd_E', 'wd_ENE', 'wd_ESE', 'wd_N', 'wd_NE', 'wd_NNE', 'wd_NNW', 'wd_NW', 'wd_S', 'wd_SE', 'wd_SSE', 'wd_SSW', 'wd_SW', 'wd_W', 'wd_WNW', 'wd_WSW']


In [21]:
from pathlib import Path

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "aotizhongxin_ready_for_prediction.csv"

raw_df.to_csv(output_path)

print("Inference-ready dataset saved:", output_path)
print("Saved successfully!")


Inference-ready dataset saved: ../data/processed/aotizhongxin_ready_for_prediction.csv
Saved successfully!


In [22]:
check_df = pd.read_csv(
    "../data/processed/aotizhongxin_ready_for_prediction.csv"
)

print("Saved file shape:", check_df.shape)
print("Verification successful!")


Saved file shape: (35064, 53)
Verification successful!
